In [ ]:
%pip install databricks-sdk pyyaml

# Provisioned Throughput — Cost Calculator & Deployment

This notebook does two things:
1. **Calculate the saturation point** — the token volume at which provisioned throughput (PT) becomes cost-competitive with pay-per-token (PPT)
2. **Deploy** — create the PT endpoint via the Databricks SDK once you've validated the economics

Configure `config.yml` before running:
- `deployment.entity_name` — Unity Catalog path to the model (`catalog.schema.model_name`)
- `deployment.provisioned_model_units` — number of model units to reserve
- `deployment.burst_scaling_enabled` — allow bursting beyond provisioned capacity

## Cost Competitiveness Calculator

PT is billed **by the hour** regardless of actual usage. PPT is billed **per token consumed**.
The goal is to find the **saturation point** — the utilization % at which PT becomes cost-competitive —
and compare it against the **PPT throughput ceiling** to determine which serving mode makes sense.

### Example: GPT OSS 120B

| | Pay-Per-Token (PPT) | Provisioned Throughput (PT) |
|---|---|---|
| Input cost | $0.15 / 1M tokens | — |
| Output cost | $0.60 / 1M tokens | — |
| Hourly cost | — | $5.00 / hr |
| Max throughput | 3,500 tokens/sec | 11,000 tokens/sec |

### The math (assuming 85% input / 15% output ratio)

```
# Blended PPT cost weighted by input/output ratio
ppt_blended_cost_per_million = (0.85 × $0.15) + (0.15 × $0.60) = $0.2175 / M tokens

# Maximum tokens each mode can process in one hour
ppt_max_tokens_per_hour = 3,500  × 3,600 =  12.6M tokens/hr  (hard ceiling)
pt_max_tokens_per_hour  = 11,000 × 3,600 =  39.6M tokens/hr

# PPT ceiling as a % of PT capacity
ppt_ceiling_pct = 12.6M / 39.6M ≈ 32%

# Cost break-even: solve for the token volume where PT hourly cost = PPT token cost
saturation_tokens = ($5.00 × 1,000,000) / $0.2175 ≈ 23M tokens/hr

# Express as a % of PT's max capacity
saturation_pct = 23M / 39.6M ≈ 58%
```

### Interpreting the result

The saturation point (**58%**) is above the PPT throughput ceiling (**32%**).
This means PT never becomes cheaper within the range where PPT is even viable —
PPT wins on cost at every throughput level it can serve.

The real decision gate is therefore **not cost, but throughput need**:

| Expected peak tokens/sec | Recommendation |
|---|---|
| ≤ 3,500 | Use PPT — always cheaper for this model |
| > 3,500 | Use PT — PPT cannot serve this load regardless of cost |

> **Note:** Replace the inputs above with actual values for your model and adjust the
> input/output ratio to match your workload before using this to make a provisioning decision.

In [ ]:
import yaml

with open("config.yml") as f:
    config = yaml.safe_load(f)

DATABRICKS_HOST      = config["databricks"]["host"]
SECRETS_SCOPE        = config["databricks"]["secrets_scope"]
TOKEN_SECRET_KEY     = config["databricks"]["token_secret_key"]
ENDPOINT_NAME        = config["model_serving"]["provisioned_endpoint"]

d = config["deployment"]
ENTITY_NAME              = d["entity_name"]
ENTITY_VERSION           = d["entity_version"]          # may be None
PROVISIONED_MODEL_UNITS  = d["provisioned_model_units"]
BURST_SCALING_ENABLED    = d["burst_scaling_enabled"]
DESCRIPTION              = d.get("description")
TAGS                     = d.get("tags", {})

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointTag

w = WorkspaceClient(
    host=DATABRICKS_HOST,
    token=dbutils.secrets.get(scope=SECRETS_SCOPE, key=TOKEN_SECRET_KEY),
)

## Step 1 — Determine your model unit count

Use the **[Databricks GenAI Pricing Calculator](https://www.databricks.com/product/pricing/genai-pricing-calculator)** to get a model unit recommendation for your workload. Enter your model, cloud/region, average input tokens per request, average output tokens per request, and QPM at your P75–P90 load level.

Set the result as `deployment.provisioned_model_units` in `config.yml` before running Step 2.

## Step 2 — Deploy the provisioned throughput endpoint

## Step 2 — Deploy the provisioned throughput endpoint

In [ ]:
endpoint = deploy_provisioned_throughput_endpoint(
    endpoint_name=ENDPOINT_NAME,
    entity_name=ENTITY_NAME,
    provisioned_model_units=PROVISIONED_MODEL_UNITS,
    entity_version=ENTITY_VERSION,
    burst_scaling_enabled=BURST_SCALING_ENABLED,
    description=DESCRIPTION,
    tags=TAGS,
)